# OpsPilot: a runbook-grounded incident copilot
**Use case (from the use-case sheet):** an assistant that
reads a Kubernetes, Jenkins or Terraform failure log, finds the matching runbook, explains the root cause with the exact log
lines, and proposes a fix, while never running or recommending a state-changing command without a human approving it.

**What you will practise:** secret redaction, context engineering (log excerpting), header-aware chunking, hybrid retrieval
(BM25 + vectors fused with RRF), small-to-big retrieval, structured output, a LangGraph with conditional routing,
`interrupt()` human-in-the-loop with a checkpointer, and an evaluation harness.

Cells marked **LIVE** call OpenAI; everything else runs offline.

## 0. Setup
Runs in Google Colab or locally. In Colab the first cell clones the repo and installs this project's requirements; locally, start Jupyter from the project folder.

In [ ]:
import os, sys, subprocess
from pathlib import Path

PROJECT = "04-use-case-projects/opspilot-devops-copilot"
if "google.colab" in sys.modules:
    if not Path("genai-agentic-ai-training").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tarang16/genai-agentic-ai-training.git"], check=True)
    os.chdir(f"genai-agentic-ai-training/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path(PROJECT).exists():  # started from the repo root
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
# LIVE: only needed for the cells marked LIVE. Never paste a key into a cell; getpass keeps it out of the file.
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## 1. The data
Seven runbooks (Markdown with front matter) and nine real-looking incident logs. One incident (Kafka lag) is deliberately *not* covered by any runbook: the right answer there is to escalate, not to guess.

In [ ]:
from opspilot.config import RUNBOOK_DIR, INCIDENT_DIR
print(sorted(p.name for p in RUNBOOK_DIR.glob("*.md")))
print(sorted(p.name for p in INCIDENT_DIR.glob("*.log")))
print((INCIDENT_DIR / "06_jenkins_scm_auth.log").read_text()[-400:])

## 2. Deterministic first: redact, detect, excerpt
Anything a regex can do reliably should not cost tokens. Note the Nexus password and AWS key in the Jenkins log: they are gone before any model or vector store sees the text.

In [ ]:
from opspilot.signals import redact, detect_signals, detect_platform, excerpt
log = (INCIDENT_DIR / "06_jenkins_scm_auth.log").read_text()
clean, n = redact(log)
print(f"{n} secrets redacted"); print(clean[-200:])
sig = detect_signals(clean)
print("platform:", detect_platform(clean, sig)); print([s.id for s in sig])
print(excerpt(clean, sig))

## 3. Chunking and hybrid retrieval
One chunk per `##` section, ids like `k8s-oomkilled.md#remediation` that the model must cite. BM25 alone first (no API key), then `expand()` pulls in sibling sections (small-to-big).

In [ ]:
from opspilot.retrieval import RunbookIndex, load_runbooks
chunks = load_runbooks(RUNBOOK_DIR)
print(len(chunks), "chunks;", chunks[0].id)
bm25_only = RunbookIndex(chunks)
hits = bm25_only.search("k8s_oomkilled Reason: OOMKilled Exit Code: 137", k=2, platform="kubernetes")
print("search:", [c.id for c, _ in hits])
print("expand:", [c.id for c, _ in bm25_only.expand(hits)])

**Exercise:** run `python evaluate.py --offline` in a terminal. Which incidents would fail retrieval if you removed the signal ids from the BM25 text in `RunbookIndex.__init__`?

## 4. The graph

In [ ]:
from opspilot import build_graph
from opspilot.config import get_llm, get_embeddings
# LIVE: hybrid index (OpenAI embeddings) + gpt-4o-mini
index = RunbookIndex(chunks, embeddings=get_embeddings())
app = build_graph(index, get_llm())
print(app.get_graph().draw_mermaid())

## 5. Run an incident end to end, including the approval gate

In [ ]:
# LIVE
import uuid
from langgraph.types import Command
from opspilot import render

cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}
state = app.invoke({"log": (INCIDENT_DIR / "01_payments_oomkilled.log").read_text()}, cfg)
if "__interrupt__" in state:
    print("PAUSED for approval:", state["__interrupt__"][0].value["commands"])
    state = app.invoke(Command(resume={"approved": False, "approver": "notebook"}), cfg)
print(render(state))

In [ ]:
# LIVE: the uncovered incident should escalate, not invent a Kafka fix
state = app.invoke({"log": (INCIDENT_DIR / "08_kafka_consumer_lag.log").read_text()},
                   {"configurable": {"thread_id": str(uuid.uuid4())}})
print(state["status"], state.get("escalation_reasons"))

## 6. Evaluate
`python evaluate.py` scores signal recall, platform accuracy, retrieval hit@3, escalation accuracy, citation accuracy and root-cause keyword hits over the golden set in `data/eval/golden.jsonl`.

In [ ]:
# LIVE
import subprocess, sys
print(subprocess.run([sys.executable, "evaluate.py"], capture_output=True, text=True).stdout)

## 7. Take it further
1. Add a runbook for the Kafka incident and a golden row; re-run the eval.
2. Replace `InMemorySaver` with `SqliteSaver` so approvals survive a restart.
3. Turn on LangSmith (`LANGSMITH_TRACING=true`) and compare token use with and without `excerpt()`.
4. Feed it your own runbooks and Jenkins console logs (redact first).